# 07b · Fine-Tuning (Unfrozen Backbone)

**Reads `folds_infection.csv` from notebook 04. Same protocol as notebook 07.**

Notebook 07 reports AUROC from a frozen ImageNet backbone with a trained linear
head, stated there as a floor rather than a tuned ceiling. This notebook tests
that directly: identical folds, identical scoring protocol, identical metrics.
The only difference is that the EfficientNet-B0 backbone is unfrozen and trained
end to end.

## Why this is a fair comparison
Everything except the one variable under test is held constant. Same
patient-grouped folds from notebook 04. Same per-photograph aggregation before
scoring, asserted rather than assumed. Same metric set. If the backbone was the
constraint, AUROC rises here. If it was not, this and notebook 07 land close
together, which is itself informative and consistent with the tissue-pathway
ablation showing the bottleneck is not model capacity.

## Checkpointed per fold
Fine-tuning the full backbone is expensive. Each fold's predictions are cached to
`outputs/finetune_ckpt/fold{k}_pred.npz` as soon as it completes. A resumed run
skips any fold whose checkpoint already exists, so an interruption costs at most
one fold, not the whole run.

## This is heavy compute
Has a Colab companion, `07b_finetune_colab.ipynb`, for GPU execution with mixed
precision. Local run documents the logic; Colab is the practical execution path
for the full 5-fold run.

## Output
`results_infection_finetuned.json`, compared against notebook 07's
`results_infection.json` if present.


In [ ]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')

INTERIM  = Path('data/interim')
OUT      = Path('outputs'); OUT.mkdir(exist_ok=True)
CKPT_DIR = OUT / 'finetune_ckpt'; CKPT_DIR.mkdir(exist_ok=True)

INF_FOLDS = INTERIM / 'folds_infection.csv'
FROZEN_RESULTS = OUT / 'results_infection.json'   # notebook 07's baseline

SEED, INPUT_SIZE, N_FOLDS = 42, 224, 5
BATCH, EPOCHS, LR, PATIENCE = 32, 8, 3e-5, 3

if not INF_FOLDS.exists():
    print(f'STOPPING. {INF_FOLDS} not found. Run notebook 04 first.')
    raise SystemExit(1)
if not FROZEN_RESULTS.exists():
    print(f'WARNING: {FROZEN_RESULTS} not found. Run notebook 07 first for')
    print('  the side-by-side comparison. This notebook will still run.')

inf = pd.read_csv(INF_FOLDS)
print(f'{len(inf):,} images, {inf.photo_unit.nunique():,} units, '
      f'{inf.patient_id.nunique():,} patients')
print(f'this notebook trains the SAME architecture on the SAME folds as')
print(f'notebook 07, with only the backbone unfrozen. Everything else is')
print(f'identical, so the comparison isolates one variable.')

In [ ]:
# Cell 2 · device, dataset, model
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEV = ('cuda' if torch.cuda.is_available()
       else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEV)

_MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
_STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

class ImgDS(Dataset):
    # num_workers=0 in the DataLoader below: workers spawn processes that
    # cannot unpickle a class defined in a notebook (__main__ problem).
    def __init__(s, paths, labels):
        s.p, s.y = list(paths), np.asarray(labels, dtype=np.float32)
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        try:
            with Image.open(s.p[i]) as im:
                im = im.convert('RGB').resize((INPUT_SIZE, INPUT_SIZE))
            a = torch.from_numpy(np.asarray(im, dtype=np.float32)/255.0)
            x = (a.permute(2,0,1) - _MEAN) / _STD
        except Exception:
            x = torch.zeros(3, INPUT_SIZE, INPUT_SIZE)
        return x, s.y[i]

def build_model():
    # full EfficientNet-B0, UNFROZEN. This is the only structural
    # difference from notebook 07's frozen linear probe.
    m = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 1))
    return m.to(DEV)

print('dataset and model builder ready (backbone unfrozen)')

In [ ]:
# Cell 3 · train one fold, with per-fold checkpointing
# Fine-tuning the full backbone is expensive, so each fold's predictions
# are cached to disk as soon as it finishes. A resumed run skips any fold
# whose checkpoint already exists rather than restarting from fold 0.
import time

def run_fold(k):
    ckpt = CKPT_DIR / f'fold{k}_pred.npz'
    if ckpt.exists():
        d = np.load(ckpt)
        print(f'  fold {k}: cached ({len(d["units"])} units)')
        return d['units'], d['probs'], d['y']

    te = inf[inf.fold == k]
    tr_all = inf[inf.fold != k]
    val_units = rng_fold.choice(tr_all.photo_unit.unique(),
                                size=max(1, tr_all.photo_unit.nunique()//6),
                                replace=False)
    va = tr_all[tr_all.photo_unit.isin(val_units)]
    tr = tr_all[~tr_all.photo_unit.isin(val_units)]

    dl_tr = DataLoader(ImgDS(tr.path, tr.label), batch_size=BATCH,
                       shuffle=True, num_workers=0)
    dl_va = DataLoader(ImgDS(va.path, va.label), batch_size=BATCH,
                       shuffle=False, num_workers=0)

    net = build_model()
    # dtype explicit: numpy integer division yields float64, and MPS
    # rejects float64 tensors outright.
    pos_w = torch.tensor(
        [(tr.label == 0).sum() / max((tr.label == 1).sum(), 1)],
        dtype=torch.float32).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    use_amp = (DEV == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    from sklearn.metrics import roc_auc_score
    best, state, wait = -1.0, None, 0
    t0 = time.time()
    for ep in range(EPOCHS):
        net.train()
        for x, y in dl_tr:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(
                    net(x).squeeze(1), y, pos_weight=pos_w)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        net.eval(); pv, yv = [], []
        with torch.no_grad():
            for x, y in dl_va:
                pv.append(torch.sigmoid(net(x.to(DEV)).squeeze(1)).cpu().numpy())
                yv.append(y.numpy())
        pv, yv = np.concatenate(pv), np.concatenate(yv)
        a = roc_auc_score(yv, pv) if len(np.unique(yv)) > 1 else 0.5
        print(f'    fold {k} ep{ep}  val AUROC {a:.4f}  ({time.time()-t0:.0f}s)')
        if a > best:
            best, wait = a, 0
            state = {kk: v.clone().cpu() for kk, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break

    net.load_state_dict(state); net.eval()
    dl_te = DataLoader(ImgDS(te.path, te.label), batch_size=BATCH,
                       shuffle=False, num_workers=0)
    pe = []
    with torch.no_grad():
        for x, y in dl_te:
            pe.append(torch.sigmoid(net(x.to(DEV)).squeeze(1)).cpu().numpy())
    pe = np.concatenate(pe)
    units, y_true = te.photo_unit.values, te.label.values

    np.savez(ckpt, units=units, probs=pe, y=y_true)
    print(f'  fold {k}: saved checkpoint ({time.time()-t0:.0f}s total)')
    return units, pe, y_true

rng_fold = np.random.default_rng(SEED)
print('fold runner ready (checkpoints per fold)')

In [ ]:
# Cell 4 · run all folds, aggregate per photograph
# Same protocol as notebook 07: predict per image, average to one score
# per photo_unit before computing any metric.
all_units, all_probs, all_y = [], [], []
for k in range(N_FOLDS):
    u, p, y_ = run_fold(k)
    all_units.append(u); all_probs.append(p); all_y.append(y_)

units = np.concatenate(all_units)
probs = np.concatenate(all_probs)
ytrue = np.concatenate(all_y)

df = pd.DataFrame({'unit': units, 'p': probs, 'y': ytrue})
per_unit = df.groupby('unit').agg(p=('p','mean'), y=('y','first'))
assert len(per_unit) == inf.photo_unit.nunique(), \
    'unit count mismatch after aggregation'
print(f'\naggregated to {len(per_unit):,} photographs, each scored once')

In [ ]:
# Cell 5 · score and compare against the frozen baseline
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, confusion_matrix)

yt, pp = per_unit.y.values, per_unit.p.values
pred = (pp >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0,1]).ravel()

finetuned = dict(
    name='fine-tuned (unfrozen backbone)',
    auroc=float(roc_auc_score(yt, pp)),
    auprc=float(average_precision_score(yt, pp)),
    mcc=float(matthews_corrcoef(yt, pred)),
    sensitivity=float(tp/max(tp+fn,1)),
    specificity=float(tn/max(tn+fp,1)),
    n_units=int(len(yt)))
finetuned['balanced_acc'] = (finetuned['sensitivity']+finetuned['specificity'])/2

print('FINE-TUNED RESULT')
print(f"  AUROC {finetuned['auroc']:.4f}   AUPRC {finetuned['auprc']:.4f}   "
      f"MCC {finetuned['mcc']:.4f}")
print(f"  sensitivity {finetuned['sensitivity']:.4f}   "
      f"specificity {finetuned['specificity']:.4f}")

if FROZEN_RESULTS.exists():
    frozen = json.load(open(FROZEN_RESULTS))
    frozen_best = max(frozen['configs'], key=lambda c: c['auroc'])
    delta = finetuned['auroc'] - frozen_best['auroc']
    print(f"\nfrozen linear probe (notebook 07) was AUROC "
          f"{frozen_best['auroc']:.4f}")
    print(f"change from fine-tuning: {delta:+.4f}")
    if delta > 0.02:
        print('  the frozen features were the constraint; fine-tuning helps.')
    elif delta < -0.02:
        print('  fine-tuning underperformed the frozen probe on this data')
        print('  size; likely overfitting given the corpus is small.')
    else:
        print('  little change: the ceiling is the task and the data, not')
        print('  the representation. Consistent with the tissue ablation.')
    finetuned['delta_vs_frozen'] = float(delta)
else:
    print('\nno frozen baseline found for comparison (run notebook 07)')

In [ ]:
# Cell 6 · save and summarise
with open(OUT / 'results_infection_finetuned.json', 'w') as f:
    json.dump(finetuned, f, indent=2)
print(f'wrote {(OUT / "results_infection_finetuned.json").resolve()}')

print('\n' + '=' * 58)
print('STAGE 07b COMPLETE')
print('=' * 58)
print(f'  {N_FOLDS} folds, backbone unfrozen, checkpointed per fold')
print(f'  AUROC {finetuned["auroc"]:.4f} on {finetuned["n_units"]:,} photographs')
if 'delta_vs_frozen' in finetuned:
    print(f'  change vs frozen probe: {finetuned["delta_vs_frozen"]:+.4f}')
print('\nnext: 08_gradcam.ipynb (unaffected by this notebook)')